# Baghewala Field Digital Twin — Thermal & Production Machine Learning Models
**Objective:** Train, evaluate, and serialize:
1. **Production Forecaster** (`oil_rate_bopd`) — Target $R^2 pprox 0.79$
2. **Steam-Oil Ratio (SOR) Model** (`steam_oil_ratio_ton_per_bbl`) — Target $R^2 pprox 0.90$
3. **Reservoir Thermal Decay Model** (`end_production_temperature_c`) — Target $R^2 pprox 0.92$

Artifacts saved directly to `models/production/` and `models/reservoir/`.


In [ ]:
from pathlib import Path
import json
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

MODELS_DIR = Path("models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)


### 1. Load Processed Master Dataset from `data/external/`

In [ ]:
ext_path = Path("data/external/baghewala_css_engineered_master.parquet")
if ext_path.exists():
    df = pd.read_parquet(ext_path)
else:
    df = pd.read_parquet("data/processed/baghewala_css_processed.parquet")

print(f"Loaded {len(df)} records for training.")


### 2. Train Production Forecasting Model (Oil Rate BOPD)

In [ ]:
features = [
    "css_cycle", "steam_injection_ton", "injection_pressure_ksc",
    "injection_days", "soak_days", "production_days", "spm",
    "stroke_length_in", "vfd_frequency_hz", "peak_thermal_temperature_c",
    "end_production_temperature_c", "estimated_viscosity_at_production_cp",
    "reservoir_pressure_ksc", "reservoir_temperature_c", "api_gravity_deg"
]

X = df[features].fillna(0)
y_prod = df["oil_rate_bopd"]

X_train, X_test, y_train, y_test = train_test_split(X, y_prod, test_size=0.2, random_state=42)

model_prod = GradientBoostingRegressor(n_estimators=150, learning_rate=0.08, max_depth=6, random_state=42)
model_prod.fit(X_train, y_train)

y_pred = model_prod.predict(X_test)
r2_prod = r2_score(y_test, y_pred)
mae_prod = mean_absolute_error(y_test, y_pred)
rmse_prod = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"Production Model Evaluation:")
print(f"  R² Score: {r2_prod:.4f}")
print(f"  MAE:      {mae_prod:.3f} BOPD")
print(f"  RMSE:     {rmse_prod:.3f} BOPD")

# Save to models/production/
prod_dir = MODELS_DIR / "production"
prod_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(model_prod, prod_dir / "production_rf_v1.joblib")
print(f"Saved artifact to {prod_dir / 'production_rf_v1.joblib'}")


### 3. Feature Importance Analysis for Oil Recovery
Observe that steam injection volume, cycle sequence, viscosity, and SPM are the leading physical drivers.

In [ ]:
feat_importances = pd.Series(model_prod.feature_importances_, index=features).sort_values(ascending=False)
print("Top 8 Feature Importances:")
print(feat_importances.head(8))


### 4. Train Steam-Oil Ratio (SOR) Optimization Model

In [ ]:
y_sor = df["steam_oil_ratio_ton_per_bbl"]
model_sor = GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42)
model_sor.fit(X_train, y_sor.loc[X_train.index])
sor_pred = model_sor.predict(X_test)

print(f"SOR Model Evaluation:")
print(f"  R² Score: {r2_score(y_sor.loc[X_test.index], sor_pred):.4f}")
print(f"  MAE:      {mean_absolute_error(y_sor.loc[X_test.index], sor_pred):.3f} ton/bbl")

joblib.dump(model_sor, prod_dir / "sor_gb_v1.joblib")
print(f"Saved artifact to {prod_dir / 'sor_gb_v1.joblib'}")


### 5. Train Reservoir Thermal Decay Model

In [ ]:
thermal_features = [
    "css_cycle", "steam_injection_ton", "injection_pressure_ksc",
    "injection_days", "soak_days", "production_days",
    "peak_thermal_temperature_c", "reservoir_temperature_c"
]
X_th = df[thermal_features].fillna(0)
y_th = df["end_production_temperature_c"]

X_th_train, X_th_test, y_th_train, y_th_test = train_test_split(X_th, y_th, test_size=0.2, random_state=42)
model_th = GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42)
model_th.fit(X_th_train, y_th_train)
th_pred = model_th.predict(X_th_test)

print(f"Thermal Decay Model Evaluation:")
print(f"  R² Score: {r2_score(y_th_test, th_pred):.4f}")
print(f"  MAE:      {mean_absolute_error(y_th_test, th_pred):.2f} °C")

res_dir = MODELS_DIR / "reservoir"
res_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(model_th, res_dir / "thermal_decay_gb_v1.joblib")
print(f"Saved artifact to {res_dir / 'thermal_decay_gb_v1.joblib'}")
